# 6. The composite Earth-2.0 ranking

Four independently reported scores, combined by a non-compensatory weighted
geometric mean; the habitable-zone validity discount that changed the top
of the ranking after an external audit; and a weight-sensitivity check on
how stable the "top 10" actually is.


This notebook is part of the reproducibility set for **Finding Earth 2.0 in
Distant Worlds**. It reads the same committed data every other output in this
project reads (`results/`, `data/processed/`, `data/manifests/`) and calls
the same `earth2` functions the pipeline itself calls -- nothing here is a
simplified restatement computed a different way. Run `python -m earth2 all`
first if `results/` does not exist yet.

See `docs/METHODS.md` for the full equations and `docs/LIMITATIONS.md` for
this project's stated caveats.


In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from earth2.config import RESULTS_DIR
from earth2.ranking import rank_catalogue, ScoreWeights, DEFAULT_WEIGHTS

cat = pd.read_parquet(RESULTS_DIR / "candidate_ranking.parquet")
ranked = rank_catalogue(cat)
print("Default weights:", DEFAULT_WEIGHTS.as_dict())


Default weights: {'earth_similarity': 0.35, 'conservative_habitability': 0.4, 'observational_confidence': 0.25, 'characterisation_potential': 0.0}


## Why a geometric mean, not an average

Under an arithmetic mean, an object with excellent measurements and superb
observability but zero habitable-zone consistency could still score
respectably -- its strong components would compensate for the disqualifying
one. A geometric mean cannot do that: any near-zero component drags the
whole index toward zero.


In [2]:
import numpy as np
strong_but_disqualified = {"similarity": 0.95, "habitability": 0.02, "confidence": 0.90}
arithmetic = np.mean(list(strong_but_disqualified.values()))
geometric = np.exp(np.mean(np.log(np.clip(list(strong_but_disqualified.values()), 0.01, 1))))
print(f"Arithmetic mean: {arithmetic:.3f}  (looks respectable -- wrong)")
print(f"Geometric mean:  {geometric:.3f}  (correctly dominated by the near-zero term)")


Arithmetic mean: 0.623  (looks respectable -- wrong)
Geometric mean:  0.258  (correctly dominated by the near-zero term)


## The TRAPPIST-1 case, worked through directly

TRAPPIST-1's host is 34 K below the habitable-zone model's validity floor,
so only a small fraction of its Monte Carlo temperature posterior falls
inside the model's domain. `hz_conservative_prob` is the mean *conditional*
on those valid draws -- multiplying by `hz_teff_valid_fraction` is what
turns a confident-looking conditional probability into the honest,
discounted picture.


In [3]:
t1 = ranked[ranked["hostname"] == "TRAPPIST-1"][
    ["pl_name", "hz_conservative_prob", "hz_teff_valid_fraction",
     "score_conservative_habitability", "earth2_rank"]
].sort_values("earth2_rank")
t1


,pl_name,hz_conservative_prob,hz_teff_valid_fraction,score_conservative_habitability,earth2_rank
28,TRAPPIST-1 e,1.000000,0.09325,0.090242,27
29,TRAPPIST-1 f,1.000000,0.09650,0.090840,28
32,TRAPPIST-1 g,0.997253,0.09100,0.082877,31
57,TRAPPIST-1 d,0.000000,0.09025,0.000000,56
60,TRAPPIST-1 c,0.000000,0.09100,0.000000,59
66,TRAPPIST-1 b,0.000000,0.09250,0.000000,65
90,TRAPPIST-1 h,0.000000,0.09550,0.000000,89


In [4]:
row = t1.iloc[0]
implied = row["hz_conservative_prob"] * row["hz_teff_valid_fraction"]
print(f"{row['pl_name']}: hz_conservative_prob={row['hz_conservative_prob']:.3f} "
      f"(computed from only {row['hz_teff_valid_fraction']:.1%} of the posterior)")
print(f"score_conservative_habitability includes a rocky-plausibility term too, "
      f"but the HZ*valid_fraction product alone is {implied:.3f} -- "
      f"'100% probability from 9% of draws' becomes roughly 9%, not 100%.")


TRAPPIST-1 e: hz_conservative_prob=1.000 (computed from only 9.3% of the posterior)
score_conservative_habitability includes a rocky-plausibility term too, but the HZ*valid_fraction product alone is 0.093 -- '100% probability from 9% of draws' becomes roughly 9%, not 100%.


## Top 10, as computed

Never hand-selected -- this is `ranked` sorted by `earth2_rank`, controls
excluded from the numbering (see `docs/LIMITATIONS.md` for why Earth still
scores only ~0.56 on observational confidence despite being, obviously, the
best-characterised planet there is).


In [5]:
cols = ["pl_name", "hostname", "earth2_index", "score_earth_similarity",
        "score_conservative_habitability", "score_observational_confidence", "mass_class"]
ranked[~ranked["is_control"]].sort_values("earth2_rank").head(10)[cols]


,pl_name,hostname,earth2_index,score_earth_similarity,score_conservative_habitability,score_observational_confidence,mass_class
0,Proxima Cen b,Proxima Cen,0.876411,0.914933,0.946188,0.730000,msini_lower_limit
1,GJ 1061 d,GJ 1061,0.875495,0.873912,0.895973,0.845841,measured
2,GJ 1002 b,GJ 1002,0.848885,0.914229,0.945319,0.644159,msini_lower_limit
4,Wolf 1069 b,Wolf 1069,0.838956,0.899716,0.930862,0.644159,msini_lower_limit
5,Teegarden's Star c,Teegarden's Star,0.823358,0.807142,0.947846,0.675841,msini_lower_limit
6,Kepler-1649 c,Kepler-1649,0.773585,0.887504,0.898481,0.502319,inferred_mass_radius
7,GJ 1002 c,GJ 1002,0.771260,0.751448,0.883018,0.644159,msini_lower_limit
9,Kepler-1229 b,Kepler-1229,0.716099,0.819347,0.732997,0.571314,inferred_mass_radius
10,GJ 667 C f,GJ 667 C,0.702383,0.804582,0.675953,0.617493,msini_lower_limit
11,TOI-700 d,TOI-700,0.683818,0.943110,0.552442,0.613363,inferred_mass_radius


## Weight sensitivity: is the top 10 an artefact of the default weights?

Re-running the identical ranking under several plausible alternative weight
sets and checking how much the top 10 actually reshuffles is exactly the
kind of check the audit recommended and that a reader should be able to run
themselves, not take on faith.


In [6]:
weight_sets = {
    "default (0.35/0.40/0.25)": DEFAULT_WEIGHTS,
    "similarity-heavy":  ScoreWeights(earth_similarity=0.60, conservative_habitability=0.25, observational_confidence=0.15),
    "habitability-heavy": ScoreWeights(earth_similarity=0.20, conservative_habitability=0.60, observational_confidence=0.20),
    "confidence-heavy":  ScoreWeights(earth_similarity=0.25, conservative_habitability=0.25, observational_confidence=0.50),
}

top10_sets = {}
for label, weights in weight_sets.items():
    r = rank_catalogue(cat, weights=weights)
    top10_sets[label] = set(r[~r["is_control"]].nsmallest(10, "earth2_rank")["pl_name"])

baseline = top10_sets["default (0.35/0.40/0.25)"]
for label, names in top10_sets.items():
    overlap = len(names & baseline)
    print(f"{label:32s}  {overlap}/10 planets shared with the default top 10")


default (0.35/0.40/0.25)          10/10 planets shared with the default top 10
similarity-heavy                  10/10 planets shared with the default top 10
habitability-heavy                9/10 planets shared with the default top 10
confidence-heavy                  9/10 planets shared with the default top 10


In [7]:
always_top10 = set.intersection(*top10_sets.values())
print(f"{len(always_top10)} planets are in the top 10 under EVERY weight set tried:")
for n in sorted(always_top10):
    print(" -", n)


9 planets are in the top 10 under EVERY weight set tried:
 - GJ 1002 b
 - GJ 1002 c
 - GJ 1061 d
 - GJ 667 C f
 - Kepler-1229 b
 - Kepler-1649 c
 - Proxima Cen b
 - Teegarden's Star c
 - Wolf 1069 b
